# Energy Sentinel — Preprocessing Pipeline

This notebook applies the cleaning pipeline to the resampled UK-DALE House 1 data.
Goals:
- Handle missing values per appliance type
- Remove sensor baseline noise (1W threshold)
- Save cleaned data to data/processed/ in parquet format

# 1 - Imports & Setup

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config.settings import settings
from src.data.loader import UKDALELoader
from src.preprocessing.cleaner import DataCleaner
from src.utils.logger import get_logger

logger = get_logger("preprocessing")

print("Imports successful")

Imports successful


# 2 - Load & Resample Raw Data

In [3]:
# Load resampled parquet — no need to reload raw data
input_path = settings.PROCESSED_DIR / "house_1_resampled_2013.parquet"

df_combined = pd.read_parquet(input_path)

print(f"Loaded: {input_path}")
print(f"Shape: {df_combined.shape}")
print(f"Date range: {df_combined.index.min()} → {df_combined.index.max()}")

Loaded: D:\Github\energy-sentinel\data\processed\house_1_resampled_2013.parquet
Shape: (525600, 7)
Date range: 2013-01-01 00:00:00+00:00 → 2013-12-31 23:59:00+00:00


# 3 - Apply Cleaning Pipeline

In [4]:
cleaner = DataCleaner()
df_cleaned = cleaner.clean(df_combined)

print(f"\nShape before: {df_combined.shape}")
print(f"Shape after : {df_cleaned.shape}")

2026-07-12 05:57:36 | INFO     | src.preprocessing.cleaner | Starting data cleaning pipeline...
2026-07-12 05:57:36 | INFO     | src.preprocessing.cleaner | Aggregate: filled 34,411 missing values with ffill
2026-07-12 05:57:36 | INFO     | src.preprocessing.cleaner | Fridge: filled 27,545 missing values with ffill
2026-07-12 05:57:36 | INFO     | src.preprocessing.cleaner | Washing Machine: filled 39,567 missing values with 0
2026-07-12 05:57:36 | INFO     | src.preprocessing.cleaner | Dishwasher: filled 26,777 missing values with 0
2026-07-12 05:57:36 | INFO     | src.preprocessing.cleaner | TV: filled 27,046 missing values with 0
2026-07-12 05:57:36 | INFO     | src.preprocessing.cleaner | Kettle: filled 34,036 missing values with 0
2026-07-12 05:57:36 | INFO     | src.preprocessing.cleaner | Microwave: filled 26,791 missing values with 0
2026-07-12 05:57:36 | INFO     | src.preprocessing.cleaner | Washing Machine: set 2,329 below-threshold readings to 0
2026-07-12 05:57:36 | INFO  

# 4 - Verify Cleaned Data

In [5]:
# Missing values after cleaning
print("Missing values after cleaning:")
print(df_cleaned.isnull().sum())

print(f"\nSummary statistics after cleaning:\n")
print(df_cleaned.describe().round(2))

Missing values after cleaning:
Aggregate          0
Washing Machine    0
Dishwasher         0
TV                 0
Kettle             0
Fridge             0
Microwave          0
dtype: int64

Summary statistics after cleaning:

       Aggregate  Washing Machine  Dishwasher         TV     Kettle  \
count  525600.00        525600.00   525600.00  525600.00  525600.00   
mean      366.07            23.31       14.84       9.62      12.57   
std       398.52           178.25      172.01      30.60     154.10   
min        76.60             0.00        0.00       0.00       0.00   
25%       185.80             0.00        0.00       0.00       0.00   
50%       246.76             0.00        0.00       0.00       0.00   
75%       399.33             0.00        0.00       0.00       0.00   
max      6138.33          2246.33     2550.75     301.20    2448.33   

          Fridge  Microwave  
count  525600.00  525600.00  
mean       37.89       4.65  
std        48.58      67.95  
min         

# 5 - Save Cleaned Data

In [6]:
output_path = settings.PROCESSED_DIR / "house_1_cleaned_2013.parquet"

df_cleaned.to_parquet(output_path)

print(f"Saved: {output_path}")
print(f"Shape: {df_cleaned.shape}")

Saved: D:\Github\energy-sentinel\data\processed\house_1_cleaned_2013.parquet
Shape: (525600, 7)
